In [4]:
import requests
from google.colab import userdata

KEY = userdata.get("GROQ-API-KEY")
URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "openai/gpt-oss-20b"   # check Groq's current free models here: https://console.groq.com/docs/rate-limits$0

comment = "Arrived cold and 50 minutes late. Never ordering again."

response = requests.post(
    URL,
    headers={"Authorization": f"Bearer {KEY}"},
    json={
        "model": MODEL,
        "temperature": 0,
        "messages": [{
            "role": "user",
            "content": f"Label this customer comment as positive or negative. Reply with one word.\n\nComment: {comment}"
        }],
    },
)
print(response.status_code)   # 200 means it worked
print(response.json()["choices"][0]["message"]["content"])   # just the label

200
Negative


Let's turn it into a function

In [6]:
def label_comment(text):
    prompt = (
        "You label customer comments for a pizza delivery company.\n"
        "Label the comment with exactly one word: positive or negative.\n"
        "If it has praise and a complaint, label by the bigger problem.\n"
        "Sarcasm counts as negative.\n\n"
        f"Comment: {text}"
    )
    response = requests.post(
        URL,
        headers={"Authorization": f"Bearer {KEY}"},
        json={"model": MODEL, "temperature": 0,
              "messages": [{"role": "user", "content": prompt}]},
    )
    answer = response.json()["choices"][0]["message"]["content"]
    return answer.strip().lower()


In [7]:
print(label_comment("Love the new stuffed crust!"))

positive


In [8]:
import pandas as pd

df = pd.read_csv("pizza_comments.csv")
df["label"] = df["comment"].apply(label_comment)
df


,id,comment,label
0,1,Arrived cold and 50 minutes late. Never orderi...,negative
1,2,Best margherita I've had in years. The crust w...,positive
2,3,$24 for a medium pizza and two breadsticks? Wa...,negative
3,4,"The pizza was amazing, too bad it showed up an...",negative
4,5,Driver was super friendly and even called to s...,negative
5,6,Great deal on the Tuesday special! Food was ju...,negative
6,7,My order was missing the garlic dip I paid for.,negative
7,8,"The app crashed when I tried to pay, so I had ...",negative
8,9,Fine.,negative
9,10,"Great, another cold pizza. Thanks a lot.",negative


In [10]:
df["label"].value_counts()

,count
label,
negative,16
positive,4


Check: does the model break the rules?

In [11]:
df["label"].unique()

array(['negative', 'positive'], dtype=object)

Even better, create a fucntion to check:

In [14]:
ALLOWED = ["positive", "negative"]

def safe_label(text):
    label = label_comment(text)
    return label if label in ALLOWED else "unclear"

And even better - add the check in the labelling function itself!

In [12]:
ALLOWED = ["positive", "negative"]

def label_comment(text):
    prompt = (
        "You label customer comments for a pizza delivery company.\n"
        "Label the comment with exactly one word: positive or negative.\n"
        "If it has praise and a complaint, label by the bigger problem.\n"
        "Sarcasm counts as negative.\n\n"
        f"Comment: {text}"
    )
    response = requests.post(
        URL,
        headers={"Authorization": f"Bearer {KEY}"},
        json={"model": MODEL, "temperature": 0,
              "messages": [{"role": "user", "content": prompt}]},
    )
    answer = response.json()["choices"][0]["message"]["content"]
    label = answer.strip().lower()
    return label if label in ALLOWED else "unclear"

In [13]:
df["label"] = df["comment"].apply(label_comment)
df

,id,comment,label
0,1,Arrived cold and 50 minutes late. Never orderi...,negative
1,2,Best margherita I've had in years. The crust w...,positive
2,3,$24 for a medium pizza and two breadsticks? Wa...,negative
3,4,"The pizza was amazing, too bad it showed up an...",negative
4,5,Driver was super friendly and even called to s...,negative
5,6,Great deal on the Tuesday special! Food was ju...,negative
6,7,My order was missing the garlic dip I paid for.,negative
7,8,"The app crashed when I tried to pay, so I had ...",negative
8,9,Fine.,negative
9,10,"Great, another cold pizza. Thanks a lot.",negative
